### CCR - Contextual Compression retriever

In [11]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from  langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv,find_dotenv

load_dotenv(find_dotenv())



True

In [4]:
docs = [
    Document(
        page_content=(
            "Our company was founded in 2011 and now has offices in three countries. "
            "The cafeteria serves lunch from noon to two, and Friday is pizza day. "
            "Refunds are issued within 14 days of purchase if the item is unused and in original packaging. "
            "The parking lot behind building B is reserved for visitors. "
            "Employees can book meeting rooms through the shared calendar."
        ),
        metadata={"id": 1, "source": "company-handbook", "needle": "Refunds are issued within 14 days of purchase"},
    ),
    Document(
        page_content=(
            "The Pacific Ocean is the largest and deepest ocean on Earth. "
            "Sea turtles can hold their breath for several hours when resting. "
            "The Mariana Trench reaches a depth of nearly 11,000 metres, deeper than Everest is tall. "
            "Coral reefs support about a quarter of all marine species. "
            "Tides are caused mainly by the gravitational pull of the Moon."
        ),
        metadata={"id": 2, "source": "ocean-facts", "needle": "The Mariana Trench reaches a depth of nearly 11,000 metres"},
    ),
    Document(
        page_content=(
            "Our mobile app supports both light and dark themes. "
            "To reset your password, open Settings, tap Security, and choose Reset Password; a link is emailed to you. "
            "Push notifications can be muted for up to 24 hours. "
            "The app is available in 12 languages. "
            "Version 4.2 added offline mode for saved articles."
        ),
        metadata={"id": 3, "source": "app-help", "needle": "open Settings, tap Security, and choose Reset Password"},
    ),
    Document(
        page_content=(
            "Photosynthesis happens in the chloroplasts of plant cells. "
            "Autumn leaves change colour because chlorophyll breaks down and reveals other pigments. "
            "Bamboo is technically a grass and can grow almost a metre in a single day. "
            "Cacti store water in thick stems to survive in dry climates. "
            "Moss has no true roots and absorbs water directly through its leaves."
        ),
        metadata={"id": 4, "source": "botany-notes", "needle": "Bamboo is technically a grass and can grow almost a metre in a single day"},
    ),
    Document(
        page_content=(
            "The 2018 football season ended with a dramatic penalty shootout. "
            "Marathon runners often hit a wall around kilometre 32 as glycogen stores run out. "
            "Basketball courts are 28 metres long in international play. "
            "Swimmers shave their bodies to reduce drag before major races. "
            "Cricket test matches can last up to five days."
        ),
        metadata={"id": 5, "source": "sports-trivia", "needle": "Marathon runners often hit a wall around kilometre 32"},
    ),
    Document(
        page_content=(
            "Our Pro plan costs 20 dollars per month and includes unlimited projects. "
            "The Free plan is limited to three projects and 1 GB of storage. "
            "Annual billing gives a 15 percent discount over monthly billing. "
            "Enterprise customers get single sign-on and a dedicated account manager. "
            "Invoices are sent on the first of every month."
        ),
        metadata={"id": 6, "source": "pricing-page", "needle": "Our Pro plan costs 20 dollars per month"},
    ),
    Document(
        page_content=(
            "Neural networks are trained by adjusting weights to minimise a loss function. "
            "Overfitting happens when a model memorises training data and performs poorly on new data. "
            "Dropout randomly disables neurons during training to reduce overfitting. "
            "GPUs speed up training because they perform many matrix operations in parallel. "
            "Learning rate schedules gradually lower the step size as training progresses."
        ),
        metadata={"id": 7, "source": "ml-notes", "needle": "Dropout randomly disables neurons during training to reduce overfitting"},
    ),
    Document(
        page_content=(
            "Rome was founded, according to legend, in 753 BC. "
            "The Colosseum could hold roughly fifty thousand spectators. "
            "Roman roads were built in layers of stone, gravel and sand for durability. "
            "Julius Caesar was assassinated on the Ides of March in 44 BC. "
            "Aqueducts carried fresh water into Roman cities from distant springs."
        ),
        metadata={"id": 8, "source": "history-notes", "needle": "Julius Caesar was assassinated on the Ides of March in 44 BC"},
    ),
    Document(
        page_content=(
            "Espresso is brewed by forcing hot water through finely ground coffee under pressure. "
            "Cold brew steeps coarse grounds in room-temperature water for 12 to 18 hours. "
            "Our shop closes at 6 pm on weekdays and 8 pm on weekends. "
            "Oat milk froths better when it is labelled barista edition. "
            "Decaf coffee still contains a small amount of caffeine."
        ),
        metadata={"id": 9, "source": "coffee-shop-faq", "needle": "Cold brew steeps coarse grounds in room-temperature water for 12 to 18 hours"},
    ),
    Document(
        # A fully irrelevant doc for most queries: good for testing filters that DROP whole chunks
        page_content=(
            "Knitting a scarf starts with casting on about forty stitches. "
            "Garter stitch is made by knitting every row, which creates a bumpy texture. "
            "Merino wool is soft and works well for beginners. "
            "Always block your finished piece to even out the stitches."
        ),
        metadata={"id": 10, "source": "knitting-guide", "needle": None},
    )
]
 
# # Each query has one `expected_doc_id`. Check two things:
# #  1) the right doc is retrieved,
# #  2) after compression, ONLY the needle sentence (or close to it) is left.
# queries = [
#     {"query": "How long do I have to return an item?",        "expected_doc_id": 1},
#     {"query": "How deep is the deepest part of the ocean?",    "expected_doc_id": 2},
#     {"query": "How do I change my password in the app?",       "expected_doc_id": 3},
#     {"query": "How fast does bamboo grow?",                    "expected_doc_id": 4},
#     {"query": "Why do marathon runners get tired late in the race?", "expected_doc_id": 5},
#     {"query": "How much is the Pro plan?",                     "expected_doc_id": 6},
#     {"query": "How can I stop my model from overfitting?",     "expected_doc_id": 7},
#     {"query": "When did Julius Caesar die?",                   "expected_doc_id": 8},
#     {"query": "How long should I steep cold brew?",            "expected_doc_id": 9},
# ]

In [8]:
embeddings=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [12]:
llm=ChatGoogleGenerativeAI(
    model="gemini-3.7-flash"
)
compressor=LLMChainExtractor.from_llm(
    llm
)

In [13]:
vector_stores=FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

In [10]:
similarity=vector_stores.as_retriever(search_type="similarity",kwargs={"k":2})


In [15]:
ccr=ContextualCompressionRetriever(
    base_retriever=similarity,
    base_compressor=compressor
)

In [16]:
query="How long do I have to return an item?"
compressed_results=ccr.invoke("query")
compressed_results

GoogleGenerativeAIError: Error embedding content: [Errno 11001] getaddrinfo failed